# Classificação de Plásticos com Keras

Este Notebook foi gerado para você rodar facilmente no **Google Colab** utilizando a sua TPU ou GPU grátis.

## Passo 1: Fazer o Upload do Dataset
Compacte a sua pasta `WaDaBa` em um arquivo `WaDaBa.zip`.
Faça o upload do `WaDaBa.zip` aqui no Colab (menu lateral de Arquivos) e rode a célula abaixo para descompactar:

In [ ]:
!unzip -q WaDaBa.zip -d /content/
!ls /content/WaDaBa

## Passo 2: Treinar os Modelos
O código abaixo é exatamente o mesmo do seu `train_models.py`, configurado para rodar os treinamentos da CNN e do Transfer Learning.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import MobileNetV2

# O caminho no Google Colab após o unzip será:
DATA_DIR = "/content/WaDaBa"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30

def create_datasets(data_dir):
    print("Carregando datasets...")
    train_dataset = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )

    val_dataset = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )
    
    class_names = train_dataset.class_names
    print(f"Classes encontradas: {class_names}")

    AUTOTUNE = tf.data.AUTOTUNE
    train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
    val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)
    
    return train_dataset, val_dataset, class_names

def build_cnn_model(num_classes):
    model = models.Sequential([
        layers.InputLayer(shape=IMG_SIZE + (3,)),
        layers.Rescaling(1./255),
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.2),
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def build_transfer_learning_model(num_classes):
    data_augmentation = models.Sequential([
        layers.RandomFlip('horizontal_and_vertical'),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.2),
    ])
    base_model = MobileNetV2(input_shape=IMG_SIZE + (3,),
                             include_top=False,
                             weights='imagenet')
    base_model.trainable = False
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = data_augmentation(inputs)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def plot_history(history, title):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc))
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Treino Acc')
    plt.plot(epochs_range, val_acc, label='Validação Acc')
    plt.legend(loc='lower right')
    plt.title(f'Acurácia ({title})')
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Treino Loss')
    plt.plot(epochs_range, val_loss, label='Validação Loss')
    plt.legend(loc='upper right')
    plt.title(f'Loss / Perda ({title})')
    plt.show()

def evaluate_model(model, val_dataset, class_names, title):
    print(f"\n--- Avaliação Detalhada: {title} ---")
    y_true = []
    y_pred = []
    for images, labels in val_dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(np.argmax(labels.numpy(), axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    report = classification_report(y_true, y_pred, target_names=class_names)
    print("Relatório de Classificação:\n", report)
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Previsto')
    plt.ylabel('Real')
    plt.title(f'Matriz de Confusão - {title}')
    plt.show()

train_ds, val_ds, class_names = create_datasets(DATA_DIR)
num_classes = len(class_names)
early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

print("\n==============================")
print("TREINANDO MODELO 1: CNN Customizada")
print("==============================")
cnn_model = build_cnn_model(num_classes)
cnn_history = cnn_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[early_stopping])
cnn_model.save('modelo_cnn.keras')
plot_history(cnn_history, 'CNN')
evaluate_model(cnn_model, val_ds, class_names, 'CNN')

print("\n==============================")
print("TREINANDO MODELO 2: Transfer Learning (MobileNetV2)")
print("==============================")
tl_model = build_transfer_learning_model(num_classes)
tl_history = tl_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[early_stopping])
tl_model.save('modelo_transfer.keras')
plot_history(tl_history, 'TransferLearning')
evaluate_model(tl_model, val_ds, class_names, 'TransferLearning')

print("Treinamento Concluído!")